# Bug hunt: the answer key

Each stage shows the planted bug, what it does, and the corrected code with its numbers.

In [1]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

## Stage 1: a tiny language model

A one-layer attention model learns to predict the next token. The data are random tokens, so nothing can be predicted: the best possible loss is ln 20 = 3.0 nats.

In [2]:
V, T, D = 20, 16, 32
class TinyLM(nn.Module):
    def __init__(self, causal):
        super().__init__(); self.causal = causal
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(T, D)
        self.q, self.k, self.v, self.out = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, V)
    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1]))
        scores = self.q(x) @ self.k(x).transpose(1, 2) / D ** 0.5
        weights = scores.softmax(-1)                                              # every position reads every position
        return self.out(weights @ self.v(x))
def fit(causal, steps=500):
    torch.manual_seed(0); m = TinyLM(causal); opt = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(steps):
        ids = torch.randint(0, V, (64, T))
        loss = F.cross_entropy(m(ids)[:, :-1].reshape(-1, V), ids[:, 1:].reshape(-1)); opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()
print("final training loss:", round(fit(False), 2))

final training loss: 0.0


**The bug.** The causal mask is missing: each position can read the **next** token, which is the target, so the loss falls to nearly zero on data with nothing to learn. A loss below the entropy of the data (here ln 20 = 3.0) is impossible for an honest model. With the mask the loss stays at about 3.

In [3]:
class TinyLM(nn.Module):
    def __init__(self, causal):
        super().__init__(); self.causal = causal
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(T, D)
        self.q, self.k, self.v, self.out = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, V)
    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1]))
        scores = self.q(x) @ self.k(x).transpose(1, 2) / D ** 0.5
        if self.causal: scores = scores.masked_fill(torch.triu(torch.ones(ids.shape[1], ids.shape[1], dtype=torch.bool), 1), float("-inf"))
        return self.out(scores.softmax(-1) @ self.v(x))
print("final training loss:", round(fit(True), 2))

final training loss: 3.0


## Stage 2: scoring a sentence with GPT-2

We measure GPT-2's perplexity on an ordinary English sentence. A good small model should be somewhere around 50 to 300.

In [4]:
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "The robot put the battery in the charger because it was empty, and then it waited quietly for the light to turn green."
def perplexity(tokenizer):
    ids = tokenizer(text, return_tensors="pt")["input_ids"]
    with torch.no_grad(): loss = model(ids, labels=ids).loss.item()
    return float(np.exp(loss))
print("perplexity:", round(perplexity(tok)))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

perplexity: 132637


**The bug.** The text was tokenised with BERT's tokenizer, but the model is GPT-2: the ids index a different vocabulary, so the model reads nonsense and the perplexity is huge. The tokenizer must be the one the model was trained with; that is why `from_pretrained` for both uses the same name.

In [5]:
print("perplexity:", round(perplexity(AutoTokenizer.from_pretrained("gpt2"))))

perplexity: 34


## Stage 3: batched generation

We generate a continuation for two prompts of different lengths in one batch, and compare each with generating it alone. They should be identical (greedy decoding).

In [6]:
gpt = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tk = AutoTokenizer.from_pretrained("gpt2"); tk.pad_token = tk.eos_token; tk.padding_side = "right"
prompts = ["The battery", "After the robot had finished charging all of its batteries, it"]
def generate(batch_prompts):
    enc = tk(batch_prompts, return_tensors="pt", padding=True)
    with torch.no_grad(): out = gpt.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tk.eos_token_id)
    return [tk.decode(o[enc["input_ids"].shape[1]:]) for o in out]
alone = [generate([p])[0] for p in prompts]; together = generate(prompts)
print("batch equals single for each prompt:", [a == b for a, b in zip(alone, together)])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

batch equals single for each prompt: [False, True]


**The bug.** Padding is on the right, so the shorter prompt is followed by padding tokens and the model generates after the padding instead of after the prompt. For generation, pad on the **left** (and pass the attention mask, which `tk(...)` does), so every prompt ends at the same position.

In [7]:
tk.padding_side = "left"                                                           # pad on the left so every prompt ENDS where the new tokens begin
alone = [generate([p])[0] for p in prompts]; together = generate(prompts)
print("batch equals single for each prompt:", [a == b for a, b in zip(alone, together)])

batch equals single for each prompt: [True, True]


## Stage 4: evaluating a language model

A model that remembers its training lines is trained on 1,000 log lines and evaluated on 150 held-out lines. We report the perplexity.

In [8]:
import random, math
from collections import Counter
random.seed(0)
lines = [f"robot R-{random.randint(1, 30)} reported battery {random.randint(1, 99)} percent at charger C{random.randint(1, 5)}" for _ in range(1000)]
pool = lines + random.sample(lines, 150)                                     # the pool was assembled from two exports that overlap
random.shuffle(pool); train_lines, eval_lines = pool[:1000], pool[1000:]
def memory_perplexity(train, test):
    # a model that remembers its training lines: the next word is predicted from the whole prefix of the line (add-0.01 smoothing)
    prefix, nxt = Counter(), Counter()
    for l in train:
        w = l.split()
        for i in range(len(w)): prefix[tuple(w[:i])] += 1; nxt[tuple(w[:i + 1])] += 1
    vocab = len({x for l in train for x in l.split()}) + 1
    total, n = 0.0, 0
    for l in test:
        w = l.split()
        for i in range(len(w)): total += -math.log((nxt[tuple(w[:i + 1])] + 0.01) / (prefix[tuple(w[:i])] + 0.01 * vocab)); n += 1
    return math.exp(total / n)
print("held-out perplexity:", round(memory_perplexity(train_lines, eval_lines), 2), "| lines of the evaluation set that also occur in the training set:", sum(l in set(train_lines) for l in eval_lines))

held-out perplexity: 13.51 | lines of the evaluation set that also occur in the training set: 46


**The bug.** The evaluation set was drawn from a pool that contains duplicates of training lines (two exports of the same logs were merged), so a good share of the 'held-out' lines were seen in training and are almost free for a model that remembers. Always check the overlap between training and evaluation data (exact duplicates here; near-duplicates and shared sources in real corpora) and remove it before measuring.

In [9]:
seen = set(train_lines)
clean_eval = [l for l in eval_lines if l not in seen]
print("held-out perplexity:", round(memory_perplexity(train_lines, clean_eval), 2), "| lines of the evaluation set that also occur in the training set:", sum(l in seen for l in clean_eval), "| lines kept:", len(clean_eval))

held-out perplexity: 26.44 | lines of the evaluation set that also occur in the training set: 0 | lines kept: 104


## Stage 5: preparing chat data for fine-tuning

We build one supervised example from a question and its answer, and mark which tokens carry loss. The model should learn to write the answer and then the end-of-turn marker.

In [10]:
qtok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
question, answer = "What is the capital of France?", "Paris."
prompt_text = f"User: {question}\nAssistant: "
full_text = prompt_text + answer                                                       # hand-written template
ids = qtok(full_text)["input_ids"]; start = len(qtok(prompt_text)["input_ids"])
loss_tokens = qtok.convert_ids_to_tokens(ids[start:])
print("tokens with loss:", loss_tokens, "| the end-of-turn marker is trained:", "<|im_end|>" in loss_tokens)

tokens with loss: ['.'] | the end-of-turn marker is trained: False


**The bug.** The conversation was written by hand with an invented format (`User:` / `Assistant:`) instead of the model's own chat template. The model was trained on `<|im_start|>` markers, so this format is unfamiliar, and the example has no `<|im_end|>`: the model is never trained to stop. The loss mask is wrong too: the trailing space of `Assistant: ` merges with the first answer token, so only the final `.` carries loss. Use `tokenizer.apply_chat_template` for both training and generation.

In [11]:
official = qtok.apply_chat_template([{"role": "user", "content": question}, {"role": "assistant", "content": answer}], tokenize=False)
start_official = len(qtok(qtok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True))["input_ids"])
loss_tokens = qtok.convert_ids_to_tokens(qtok(official)["input_ids"][start_official:])
print("tokens with loss:", loss_tokens, "| the end-of-turn marker is trained:", "<|im_end|>" in loss_tokens)

tokens with loss: ['Paris', '.', '<|im_end|>', 'Ċ'] | the end-of-turn marker is trained: True
